# AML Transaction Classification – Model Training & Evaluation

**Project:** Money Laundering Detection  
**Objective:** Train and compare machine learning models to classify transactions as legitimate or potentially related to money laundering.

This notebook covers data loading, cleaning, leakage investigation, feature engineering, model training, threshold optimization, and final evaluation.


## 1. Import Libraries & Load Dataset

Import the required libraries and load the prepared dataset.


In [ ]:
# Install model libraries
%pip -q install xgboost catboost

# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Data splitting and preprocessing
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Models
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    average_precision_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

from google.colab import files
from IPython.display import display

# Upload dataset
uploaded = files.upload()

file_name = "money_laundring_cleaned_and_feature_engineered.csv"
df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
display(df.head())


## 2. Initial Data Quality Check

Inspect dataset structure, missing values, duplicates, and target distribution before cleaning.


In [ ]:
print("Dataset information:")
df.info()

print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing Values"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
display(df["Is_laundering"].value_counts(dropna=False).to_frame("Count"))

print("\nTarget percentages:")
display(
    (df["Is_laundering"].value_counts(normalize=True, dropna=False) * 100)
    .round(2)
    .to_frame("Percentage")
)


## 3. Data Cleaning

- Remove duplicate rows.
- Remove rows with missing or invalid target values.
- Ensure the target is binary and stored as an integer.
- Convert Boolean-like features to numeric values.


In [ ]:
# Remove duplicate rows
df = df.drop_duplicates().copy()

# Convert target to numeric and remove missing/invalid values
df["Is_laundering"] = pd.to_numeric(
    df["Is_laundering"], errors="coerce"
)

df = df.dropna(subset=["Is_laundering"]).copy()
df = df[df["Is_laundering"].isin([0, 1])].copy()
df["Is_laundering"] = df["Is_laundering"].astype(int)

# Convert Boolean-like columns to numeric
boolean_columns = [
    "is_payment_same",
    "is_bank_same_location"
]

for col in boolean_columns:
    if col in df.columns:
        df[col] = df[col].replace({
            True: 1,
            False: 0,
            "True": 1,
            "False": 0,
            "true": 1,
            "false": 0
        })
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Data cleaning completed.")
print("Cleaned dataset shape:", df.shape)
print("Missing target values:", df["Is_laundering"].isna().sum())
print("\nTarget distribution after cleaning:")
display(df["Is_laundering"].value_counts().to_frame("Count"))


## 4. Target Leakage Investigation

Investigate features that may reveal the target directly or indirectly.

`Laundering_type` is excluded from model features because its values can encode the laundering label. Frequency features are also excluded according to the project's feature-selection decision.


In [ ]:
# Investigate the relationship between Laundering_type and the target
if "Laundering_type" in df.columns:
    laundering_type_analysis = pd.crosstab(
        df["Laundering_type"],
        df["Is_laundering"],
        normalize="index"
    ).round(3)

    display(laundering_type_analysis)

# Check whether frequency features exist
frequency_columns = [
    "Sender_frequancy",
    "Reciever_frequancy"
]

available_frequency_features = [
    col for col in frequency_columns if col in df.columns
]

print("Available frequency features:", available_frequency_features)


## 5. Feature Engineering

Convert transaction hour into cyclical features so that hours near midnight are represented as close to each other.

Also ensure `Amount` is numeric and non-negative.


In [ ]:
# Create cyclical hour features
if "hour" in df.columns:
    df["hour"] = pd.to_numeric(df["hour"], errors="coerce")

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# Ensure Amount is numeric and non-negative
if "Amount" in df.columns:
    df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")
    df["Amount"] = df["Amount"].clip(lower=0)

print("Feature engineering completed.")
display(df.head())


## 6. Define Features & Target

Define the target column and remove columns that should not be used as model inputs.

Excluded columns:
- `Is_laundering`: target variable.
- `Laundering_type`: potential target leakage.
- `hour`: replaced by cyclical features.
- `Sender_frequancy` and `Reciever_frequancy`: excluded based on the project's feature-selection decision.


In [ ]:
target_column = "Is_laundering"

columns_to_exclude = [
    "Is_laundering",
    "Laundering_type",
    "hour",
    "Sender_frequancy",
    "Reciever_frequancy"
]

X = df.drop(
    columns=columns_to_exclude,
    errors="ignore"
).copy()

y = df[target_column].copy()

# Identify categorical and numerical features
categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)
print("Number of input features:", X.shape[1])
print("Target included in X:", target_column in X.columns)


## 7. Split Data into Train, Validation & Test Sets

Use stratified splitting to preserve the target-class proportions in all three datasets.

- Training: 60%
- Validation: 20%
- Test: 20%


In [ ]:
# First split: 80% development data and 20% final test data
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Second split: 75% of development data for training, 25% for validation
X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.25,
    random_state=42,
    stratify=y_dev
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

print("\nTraining target distribution:")
display(y_train.value_counts(normalize=True).round(3).to_frame("Proportion"))

print("\nValidation target distribution:")
display(y_val.value_counts(normalize=True).round(3).to_frame("Proportion"))

print("\nTest target distribution:")
display(y_test.value_counts(normalize=True).round(3).to_frame("Proportion"))


## 8. Build Preprocessing Pipelines

Numerical features:
- Missing values are imputed using the median.

Categorical features:
- Missing values are imputed using the most frequent value.
- Categories are converted using One-Hot Encoding.
- Unseen categories are ignored during transformation.

All preprocessing is fitted within each model pipeline using the training data only.


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created.")


## 9. Train Random Forest

Train Random Forest as the baseline model. Class weighting is used to account for class imbalance.


In [ ]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

print("Random Forest training completed.")


## 10. Train XGBoost

Train XGBoost using the same training data and preprocessing approach.


In [ ]:
xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_model.fit(X_train, y_train)

print("XGBoost training completed.")


## 11. Train CatBoost

Train CatBoost with class balancing. In this implementation, categorical variables are imputed and One-Hot Encoded before being passed to CatBoost; native categorical processing is not used here.


In [ ]:
cat_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ))
        ]), categorical_features)
    ]
)

cat_model = Pipeline(steps=[
    ("preprocessor", cat_preprocessor),
    ("classifier", CatBoostClassifier(
        iterations=800,
        depth=6,
        learning_rate=0.03,
        l2_leaf_reg=5,
        auto_class_weights="Balanced",
        eval_metric="PRAUC",
        random_seed=42,
        verbose=0,
        allow_writing_files=False
    ))
])

cat_model.fit(X_train, y_train)

print("CatBoost training completed.")


## 12. Evaluate Model Ranking Performance on Validation Data

Calculate threshold-independent metrics for all models:
- Average Precision (reported here as PR-AUC).
- ROC-AUC.

These metrics evaluate how well each model ranks positive cases above negative cases, independent of the selected classification threshold.


In [ ]:
models = {
    "Random Forest": rf_model,
    "XGBoost": xgb_model,
    "CatBoost": cat_model
}

validation_ranking_results = []
models_probabilities = {}

for model_name, trained_model in models.items():
    val_prob = trained_model.predict_proba(X_val)[:, 1]
    models_probabilities[model_name] = val_prob

    validation_ranking_results.append({
        "Model": model_name,
        "PR-AUC (Average Precision)": average_precision_score(y_val, val_prob),
        "ROC-AUC": roc_auc_score(y_val, val_prob)
    })

validation_ranking_df = pd.DataFrame(validation_ranking_results)

display(
    validation_ranking_df.style.format({
        "PR-AUC (Average Precision)": "{:.4f}",
        "ROC-AUC": "{:.4f}"
    })
)


## 13. Threshold Optimization on Validation Data

A common operating requirement is applied to all models:

**Minimum Recall = 0.75**

For each model, thresholds from 0.01 to 0.99 are evaluated. Among thresholds satisfying the Recall requirement, the selected threshold maximizes Precision; F1-score is used as a tie-breaker.

The test set is not used for threshold selection.


In [ ]:
minimum_recall = 0.75
threshold_values = np.round(np.arange(0.01, 1.00, 0.01), 2)

threshold_results = []

for model_name, probabilities in models_probabilities.items():
    for threshold in threshold_values:
        predictions = (probabilities >= threshold).astype(int)

        threshold_results.append({
            "Model": model_name,
            "Threshold": threshold,
            "Precision": precision_score(
                y_val, predictions, zero_division=0
            ),
            "Recall": recall_score(
                y_val, predictions, zero_division=0
            ),
            "F1-score": f1_score(
                y_val, predictions, zero_division=0
            )
        })

threshold_df = pd.DataFrame(threshold_results)

# Keep only thresholds satisfying the minimum Recall requirement
eligible_results = threshold_df[
    threshold_df["Recall"] >= minimum_recall
].copy()

if eligible_results.empty:
    raise ValueError(
        "No tested threshold satisfies the minimum Recall requirement."
    )

# Select the highest Precision for each model; use F1-score as tie-breaker
best_thresholds = (
    eligible_results
    .sort_values(
        ["Model", "Precision", "F1-score"],
        ascending=[True, False, False]
    )
    .groupby("Model")
    .head(1)
    .sort_values("F1-score", ascending=False)
    .reset_index(drop=True)
)

print(f"Best threshold for each model (Recall >= {minimum_recall}):")

display(
    best_thresholds.style.format({
        "Threshold": "{:.2f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-score": "{:.4f}"
    })
)


## 14. Model Comparison & Selection

Compare the best validation operating point for each model under the same minimum Recall requirement.

The model with the strongest Precision and F1-score balance is selected for final evaluation.


In [ ]:
comparison_table = best_thresholds[
    ["Model", "Threshold", "Precision", "Recall", "F1-score"]
].copy()

comparison_table = comparison_table.sort_values(
    "F1-score",
    ascending=False
).reset_index(drop=True)

display(
    comparison_table.style.format({
        "Threshold": "{:.2f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-score": "{:.4f}"
    })
)

# Select CatBoost based on the validation comparison
selected_model_name = "CatBoost"

selected_model_row = best_thresholds[
    best_thresholds["Model"] == selected_model_name
]

if selected_model_row.empty:
    raise ValueError("Selected model was not found in the threshold results.")

final_model = models[selected_model_name]
final_threshold = float(selected_model_row.iloc[0]["Threshold"])

print("Selected model:", selected_model_name)
print("Selected threshold:", final_threshold)


## 15. Final Evaluation on Test Set

Evaluate the selected CatBoost model once on the held-out test set using the threshold selected from validation.

Do not use these test results to tune the threshold or choose model parameters.


In [ ]:
# Predict probabilities on the test set
test_prob = final_model.predict_proba(X_test)[:, 1]

# Apply the threshold selected using validation data
test_pred = (test_prob >= final_threshold).astype(int)

# Calculate final test metrics
final_metrics = {
    "Model": selected_model_name,
    "Threshold": final_threshold,
    "Accuracy": accuracy_score(y_test, test_pred),
    "Precision": precision_score(y_test, test_pred, zero_division=0),
    "Recall": recall_score(y_test, test_pred, zero_division=0),
    "F1-score": f1_score(y_test, test_pred, zero_division=0),
    "PR-AUC (Average Precision)": average_precision_score(y_test, test_prob),
    "ROC-AUC": roc_auc_score(y_test, test_prob)
}

final_results = pd.DataFrame([final_metrics])

print("Final Test Results:")
display(
    final_results.style.format({
        "Threshold": "{:.2f}",
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-score": "{:.4f}",
        "PR-AUC (Average Precision)": "{:.4f}",
        "ROC-AUC": "{:.4f}"
    })
)

print("\nFinal Test Confusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nFinal Test Classification Report:")
print(classification_report(
    y_test,
    test_pred,
    target_names=["Legitimate", "Laundering"],
    digits=4,
    zero_division=0
))


## 16. Confusion Matrix – CatBoost Test Set

Visualize the number of correctly and incorrectly classified transactions.


In [ ]:
cm = confusion_matrix(y_test, test_pred)

fig, ax = plt.subplots(figsize=(7, 5))

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Legitimate", "Laundering"]
).plot(
    cmap="Blues",
    values_format="d",
    ax=ax
)

ax.set_title("CatBoost - Confusion Matrix (Test Set)")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("Actual Label")

plt.tight_layout()
plt.show()


## 17. ROC Curve – CatBoost Test Set

The ROC curve illustrates the relationship between the True Positive Rate and False Positive Rate across classification thresholds.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

RocCurveDisplay.from_predictions(
    y_test,
    test_prob,
    name="CatBoost",
    ax=ax
)

ax.set_title("CatBoost - ROC Curve (Test Set)")
plt.tight_layout()
plt.show()


## 18. Precision-Recall Curve – CatBoost Test Set

The Precision-Recall curve illustrates the trade-off between Precision and Recall across classification thresholds. It is particularly informative when the target classes are imbalanced.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

PrecisionRecallDisplay.from_predictions(
    y_test,
    test_prob,
    name="CatBoost",
    ax=ax
)

ax.set_title("CatBoost - Precision-Recall Curve (Test Set)")
plt.tight_layout()
plt.show()


## 19. Final Notes

- CatBoost was selected based on validation performance under the minimum Recall requirement.
- The selected threshold was determined using the validation set.
- The test set was used for final evaluation, not threshold optimization.
- The dataset is synthetic; performance on real-world banking transactions may differ.
- PR-AUC in this notebook is calculated using scikit-learn's `average_precision_score` (Average Precision).
